# 02 — Optical-Only Baseline (YOLOv8 on TrashCan)

Trains YOLOv8m fine-tuned on the TrashCan 1.0 optical dataset.
Outputs: trained weights, mAP50, mAP50-95, per-class precision/recall.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import shutil
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from ultralytics import YOLO

TRASHCAN_ROOT = Path('../data/trashcan')
RESULTS_DIR   = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

## Step 1 — Convert COCO annotations to YOLO format

YOLOv8 expects `images/` and `labels/` folders with `.txt` label files.
Each line: `class_id cx cy w h` (all normalized 0-1).

In [ ]:
def coco_to_yolo(ann_json: Path, img_src: Path, out_root: Path, split: str):
    """Convert COCO JSON annotations to YOLO .txt label files."""
    coco = COCO(str(ann_json))
    cats = coco.loadCats(coco.getCatIds())
    # Map category id → 0-indexed
    cat_map = {c['id']: i for i, c in enumerate(cats)}

    lbl_dir = out_root / 'labels' / split
    img_dir = out_root / 'images' / split
    lbl_dir.mkdir(parents=True, exist_ok=True)
    img_dir.mkdir(parents=True, exist_ok=True)

    for img_id, img_info in coco.imgs.items():
        W, H = img_info['width'], img_info['height']
        fname = img_info['file_name']

        # Copy image
        src = img_src / fname
        dst = img_dir / Path(fname).name
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)

        # Write label
        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)
        lbl_path = lbl_dir / (Path(fname).stem + '.txt')
        with open(lbl_path, 'w') as f:
            for ann in anns:
                x, y, w, h = ann['bbox']
                if w <= 0 or h <= 0:
                    continue
                cx = (x + w / 2) / W
                cy = (y + h / 2) / H
                nw = w / W
                nh = h / H
                cls = cat_map[ann['category_id']]
                f.write(f'{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n')

    print(f'{split}: {len(coco.imgs)} images converted')


YOLO_ROOT = TRASHCAN_ROOT / 'yolo_format'
for split in ['train', 'val', 'test']:
    ann_path = TRASHCAN_ROOT / 'annotations' / f'instances_{split}.json'
    img_path = TRASHCAN_ROOT / 'images' / split
    if ann_path.exists():
        coco_to_yolo(ann_path, img_path, YOLO_ROOT, split)

In [ ]:
# Write dataset YAML for ultralytics
import yaml

dataset_yaml = {
    'path': str(YOLO_ROOT.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc': 3,
    'names': ['trash', 'bio', 'rov'],
}

yaml_path = YOLO_ROOT / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print('Dataset YAML saved to:', yaml_path)
print(yaml.dump(dataset_yaml))

## Step 2 — Train YOLOv8m

In [ ]:
model = YOLO('yolov8m.pt')  # downloads pretrained weights on first run

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    fl_gamma=2.0,        # focal loss gamma
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.1,
    mosaic=1.0,
    mixup=0.1,
    degrees=10.0,
    project=str(RESULTS_DIR / 'optical'),
    name='yolov8m_trashcan',
    save=True,
    plots=True,
    verbose=True,
)

print('Best checkpoint:', results.save_dir)

## Step 3 — Evaluate on Test Set

In [ ]:
best_weights = Path(results.save_dir) / 'weights' / 'best.pt'
model_eval = YOLO(str(best_weights))

metrics = model_eval.val(
    data=str(yaml_path),
    split='test',
    imgsz=640,
    batch=16,
)

print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')
print('\nPer-class AP50:')
for cls, ap in zip(['trash', 'bio', 'rov'], metrics.box.ap50):
    print(f'  {cls:15s}: {ap:.4f}')

## Step 4 — Visualize Predictions

In [ ]:
from PIL import Image
import glob

test_imgs = list((YOLO_ROOT / 'images' / 'test').glob('*.jpg'))[:6]
if not test_imgs:
    test_imgs = list((YOLO_ROOT / 'images' / 'test').glob('*.png'))[:6]

if test_imgs:
    preds = model_eval.predict(source=[str(p) for p in test_imgs],
                               conf=0.25, imgsz=640, verbose=False)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, pred in zip(axes.flatten(), preds):
        img_with_boxes = pred.plot()  # returns BGR numpy array
        ax.imshow(img_with_boxes[..., ::-1])  # BGR → RGB
        ax.axis('off')
    plt.suptitle('YOLOv8m — Test Predictions (conf≥0.25)')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'optical_predictions.png', dpi=100)
    plt.show()
else:
    print('No test images found. Check YOLO_ROOT / images / test path.')

## Results Summary

| Metric | Value |
|---|---|
| mAP50 | *fill after run* |
| mAP50-95 | *fill after run* |
| Precision | *fill after run* |
| Recall | *fill after run* |

Save best checkpoint path for fusion notebook:
```python
OPTICAL_WEIGHTS = 'results/optical/yolov8m_trashcan/weights/best.pt'
```